# 🎞️ ROTBOTS — Assemble
## Videos + Audio + Credits → Final Video

---

1. Normalize clips → 2. Generate credits → 3. Concatenate → 4. Add narration → 5. Export

> **🤔** How does editing change meaning? Who gets credited?

---

In [ ]:
# SETUP
import json, subprocess, textwrap
from pathlib import Path
from IPython.display import display, Markdown, Video

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)
print('✅ Setup complete')

In [ ]:
# SELECT SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d / 'session_info.json').exists()])
if not sessions: print('❌ No sessions!')
else:
    print('📂 Sessions:')
    for i,s in enumerate(sessions): print(f'   {i}: {s}')

SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
VIDEOS_DIR = SESSION_DIR / 'videos'
AUDIO_DIR = SESSION_DIR / 'audio'

video_files = sorted(VIDEOS_DIR.glob('scene_*.mp4')) if VIDEOS_DIR.exists() else []
audio_files = sorted(AUDIO_DIR.glob('narration_*.mp3')) if AUDIO_DIR.exists() else []
essay = None
if (SESSION_DIR / 'essay_script.json').exists():
    with open(SESSION_DIR / 'essay_script.json') as f: essay = json.load(f)

print(f'✅ Session: {SESSION_NAME}')
print(f'   🎬 {len(video_files)} videos, 🎙️ {len(audio_files)} narrations')
if not video_files: print('❌ No videos!')

In [ ]:
# HELPERS
def get_dur(p):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0
def ffmpeg(cmd,desc=''):
    if desc: print(f'   {desc}...',end=' ',flush=True)
    r=subprocess.run(cmd,capture_output=True,text=True,timeout=300)
    if r.returncode==0:
        if desc: print('✅')
        return True
    print(f'❌ {r.stderr[:200]}'); return False
print('✅ Helpers loaded')

In [ ]:
# STEP 1: NORMALIZE
print('🔧 Step 1: Normalizing...')
norm = []
for v in video_files:
    out = TEMP / v.name
    if ffmpeg(['ffmpeg','-y','-i',str(v),'-vf','scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black','-r','24','-pix_fmt','yuv420p','-c:v','libx264','-preset','fast','-crf','23','-an',str(out)], f'{v.name}'):
        norm.append(out)
print(f'✅ {len(norm)} normalized')

In [ ]:
# STEP 2: CREDITS
print('🎬 Step 2: Generating credits...')
title = essay.get('title','Untitled') if essay else 'Untitled'
sources = essay.get('credits',{}).get('sources',[]) if essay else []

lines = [title, '', 'Generated by ROTBOTS', 'Content Machine Workshop', '', '— Sources —']
for s in sources: lines.append(s[:60]+'...' if len(s)>60 else s)
lines += ['', '— Pipeline —', 'LLM: Groq (Llama 3.3)', 'Video: fal.ai', 'Voice: Edge-TTS', 'Composition: FFmpeg', '', 'LI-MA TDA 2026, Amsterdam']

DUR=8; LH=42; scroll_speed=(720+len(lines)*LH)/DUR
filters = []
for i,line in enumerate(lines):
    if not line: continue
    safe = line.replace("'", "\u2019").replace(':','\\:').replace('%','%%')
    filters.append(f"drawtext=text='{safe}':fontsize=28:fontcolor=white:x=(w-text_w)/2:y=h+{i*LH}-{scroll_speed:.1f}*t")

credits_path = TEMP / 'credits.mp4'
if ffmpeg(['ffmpeg','-y','-f','lavfi','-i',f'color=c=black:s=1280x720:d={DUR}:r=24','-vf',','.join(filters),'-pix_fmt','yuv420p','-c:v','libx264','-preset','fast',str(credits_path)], 'Rolling credits'):
    print(f'   {DUR}s, {len(sources)} sources credited')
else: credits_path = None

In [ ]:
# STEP 3: CONCATENATE
print('🎬 Step 3: Concatenating...')
cf = TEMP / 'concat.txt'
with open(cf,'w') as f:
    for v in norm: f.write(f"file '{v}'\n")
    if credits_path and credits_path.exists(): f.write(f"file '{credits_path}'\n")

concat_out = TEMP / 'concatenated.mp4'
if ffmpeg(['ffmpeg','-y','-f','concat','-safe','0','-i',str(cf),'-c','copy',str(concat_out)], 'Concat'):
    print(f'   Duration: {get_dur(concat_out):.1f}s')

In [ ]:
# STEP 4: AUDIO MIX
final = SESSION_DIR / 'final_video.mp4'
if audio_files:
    print('🎙️ Step 4: Adding narration...')
    acf = TEMP / 'audio_concat.txt'
    with open(acf,'w') as f:
        for a in audio_files: f.write(f"file '{a}'\n")
    combined = TEMP / 'narration.mp3'
    ffmpeg(['ffmpeg','-y','-f','concat','-safe','0','-i',str(acf),'-c','copy',str(combined)],'Combining audio')
    ffmpeg(['ffmpeg','-y','-i',str(concat_out),'-i',str(combined),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-shortest',str(final)],'Mixing')
else:
    print('ℹ️ No narration')
    import shutil; shutil.copy2(concat_out, final)

if final.exists():
    print(f'\n✅ Final video: {get_dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')
    print(f'   📁 {final}')

---
## 🎬 Watch!

In [ ]:
# PLAY
if final.exists():
    display(Markdown(f'# 🎬 {title}\n*Generated by ROTBOTS*\n\n---'))
    try: display(Video(str(final),embed=True,width=720))
    except: print(f'Download from Google Drive: My Drive/rotbots_workshop/{SESSION_NAME}/final_video.mp4')
else: print('❌ No final video')

In [ ]:
# DOWNLOAD (optional)
DOWNLOAD = False
if DOWNLOAD and final.exists():
    from google.colab import files; files.download(str(final))
else:
    print(f'📁 Google Drive: rotbots_workshop/{SESSION_NAME}/final_video.mp4')
    print(f'   Set DOWNLOAD = True to download directly.')

---
## 🎤 Screening & Critique

- How does yours compare to others?
- Where is the bias?
- Who is the "author"?
- Could you tell it was AI?

---

In [ ]:
# PIPELINE SUMMARY
print('📊 Pipeline Summary')
print('='*50)
steps = []
if (SESSION_DIR/'summaries.json').exists():
    with open(SESSION_DIR/'summaries.json') as f: d=json.load(f)
    steps.append(f'📥 {len(d.get("sources",[] ))} sources scraped')
if essay:
    chs=len(essay.get('chapters',[])); secs=sum(len(c.get('sections',[])) for c in essay.get('chapters',[]))
    tw=sum(len(s.get('narration','').split()) for c in essay.get('chapters',[]) for s in c.get('sections',[]))
    steps.append(f'✍️ "{essay.get("title","")}" — {chs}ch, {secs}sec, {tw}w')
steps.append(f'🎬 {len(video_files)} video clips')
steps.append(f'🎙️ {len(audio_files)} narrations')
steps.append(f'📜 {len(sources)} sources credited')
if final.exists(): steps.append(f'🎞️ Final: {get_dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')
for s in steps: print(f'   {s}')
print(f'\n🤖 All from a single topic. Human input: topic + Play.')

---
## 🏁 That's it!

The pipeline: scraped sources → essay → storyboard → video prompts → voice → video clips → credits → final video.

**Every step: automated decisions about what's important, what looks good, what sounds right.**

The question isn't whether AI can make content. It's: **what does that mean?**

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026, Amsterdam*